# Target smoothing — sigma adaptivo por task

Gaussiano no-causal sobre posición con distintos sigma según task.
Todo offline; online no se aplica ningún procesamiento adicional.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataset.preprocessing import estimate_fs, compute_target_velocity

CSV        = '../data/raw/P01/P01_S02_20260525_184004.csv'
TOP_OFFSET = 500.0
BOT_OFFSET = 550.0

df    = pd.read_csv(CSV)
t     = df['t_micros'].to_numpy(np.float64) / 1e6
fs    = estimate_fs(t)
tasks = df['task'].to_numpy()
print(f'fs = {fs:.1f} Hz   n = {len(df):,}')
print(df['task'].value_counts())


In [ ]:
y_top = df['joy_ext_ind'].to_numpy()
y_bot = df['joy_flex_ind'].to_numpy()

SIGMAS = [30, 50, 80]  # ms — 50 es el actual

targets = {}
for s in SIGMAS:
    joy, vel = compute_target_velocity(
        y_top, y_bot, fs=fs, sigma_ms=s,
        top_offset=TOP_OFFSET, bottom_offset=BOT_OFFSET)
    targets[s] = (joy, vel)

COLORS = ['royalblue', 'black', 'tomato']
print('Targets OK.')


## Posición suavizada por task

In [ ]:
TASKS = [
    'free_controlled_ind', 'slow_flex_ind', 'slow_ext_ind',
    'fast_flex_ind', 'fast_ext_ind', 'holds_intermediate_ind',
    'microcorrections_ind', 'slow_cycles_ind',
]
PLOT_S = 8.0

fig, axes = plt.subplots(len(TASKS), 1, figsize=(16, 3.5 * len(TASKS)))
for ax, task in zip(axes, TASKS):
    idx = np.where(tasks == task)[0]
    if not len(idx): ax.set_title(f'{task} — sin datos'); continue
    i0 = idx[0]; i1 = min(i0 + int(PLOT_S * fs), idx[-1] + 1)
    sl = slice(i0, i1); tr = t[sl] - t[i0]
    for s, c in zip(SIGMAS, COLORS):
        joy = targets[s][0]
        ax.plot(tr, joy[sl], color=c, lw=2.0 if s==50 else 1.2,
                label=f'σ={s}ms' + (' ← actual' if s==50 else ''))
    ax.set_title(task, fontweight='bold'); ax.set_ylabel('posicion (joy)')
    ax.set_xlabel('t (s)'); ax.legend(fontsize=9, ncol=3); ax.grid(alpha=0.3)
plt.suptitle('Posición suavizada — distintos sigma (no-causal)', fontsize=13, fontweight='bold', y=1.001)
plt.tight_layout(); plt.show()


## Velocidad derivada por task

In [ ]:
fig, axes = plt.subplots(len(TASKS), 1, figsize=(16, 3.5 * len(TASKS)))
for ax, task in zip(axes, TASKS):
    idx = np.where(tasks == task)[0]
    if not len(idx): ax.set_title(f'{task} — sin datos'); continue
    i0 = idx[0]; i1 = min(i0 + int(PLOT_S * fs), idx[-1] + 1)
    sl = slice(i0, i1); tr = t[sl] - t[i0]
    for s, c in zip(SIGMAS, COLORS):
        vel = targets[s][1]
        ax.plot(tr, vel[sl], color=c, lw=2.0 if s==50 else 1.2,
                label=f'σ={s}ms' + (' ← actual' if s==50 else ''))
    ax.axhline(0, color='gray', lw=0.5, ls='--')
    ax.set_title(task, fontweight='bold'); ax.set_ylabel('velocidad (joy/s)')
    ax.set_xlabel('t (s)'); ax.legend(fontsize=9, ncol=3); ax.grid(alpha=0.3)
plt.suptitle('Velocidad — distintos sigma (no-causal)', fontsize=13, fontweight='bold', y=1.001)
plt.tight_layout(); plt.show()
